# ENGINE TARGETS — value con evaluaciones de Stockfish (RQ2)
Hipótesis: el MSE del value está estancado por la SUPERVISION (resultado de partida),
no por optimización. Entrenamos el value contra evaluaciones de engine de la DB
pública de Lichess (394M posiciones, 20.6GB zst).
**Sube antes** `checkpoints/masked-v1/best_model.pt` (panel de archivos, misma ruta).
Criterio del piloto: `loss_mse` (train) claramente < 0.5 y pearson_r(val) > 0.4.


In [ ]:
!git clone https://github.com/oscar2697/chess-transformer.git 2>/dev/null; true
%cd /content/chess-transformer
!git pull origin main && git log --oneline -1
!pip install python-chess torch matplotlib langchain-openai plotly zstandard -q
print('setup ok')


In [ ]:
from google.colab import userdata
import os
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['NVIDIA_API_KEY'] = userdata.get('NVIDIA_KEY')
os.environ['LLM_MODEL'] = 'moonshotai/kimi-k3'
from src.graph_agents.llm_agents import get_llm
print('backend:', get_llm()[0])


In [ ]:
# VERIFICA el checkpoint base (igual que value_only)
import pathlib
ck = pathlib.Path('checkpoints/masked-v1/best_model.pt')
assert ck.exists(), f'FALTA {ck}: sube tu checkpoint local antes de entrenar'
print('ckpt ok:', f'{ck.stat().st_size/1e6:.0f} MB')


In [ ]:
# Datos elite (misma semilla de dataset que masked-v1)
!mkdir -p data/raw
!wget -q https://database.nikonoel.fr/lichess_elite_2024-01.zip -O data/raw/e01.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-02.zip -O data/raw/e02.zip
!wget -q https://database.nikonoel.fr/lichess_elite_2024-03.zip -O data/raw/e03.zip
!unzip -o -q data/raw/e01.zip -d data/raw/ && unzip -o -q data/raw/e02.zip -d data/raw/ && unzip -o -q data/raw/e03.zip -d data/raw/
from src.graph_agents.llm_agents import run_agent
r = run_agent('data_engineer', 'Preprocess 3 elite months',
              overrides={'pgn_path': 'data/raw', 'elo_threshold': 2000,
                         'max_positions': 600000},
              auto=True)
print(r['result'])


In [ ]:
# Descarga de la DB de evaluaciones (20.6GB, ~5-10 min) — NO descomprimir, se lee por stream
!wget -q https://database.lichess.org/lichess_db_eval.jsonl.zst -O data/raw/lichess_db_eval.jsonl.zst
!ls -lh data/raw/lichess_db_eval.jsonl.zst


In [ ]:
# JOIN: nuestras posiciones vs 394M evals (stream, ~30-60 min CPU).
# Re-ejecutable: si muere, re-ejecutar (re-lee el stream).
from src.graph_agents.nodes.engine_targets import run_engine_targets
stats = run_engine_targets(evals_path='data/raw/lichess_db_eval.jsonl.zst')
print(stats)
# Coverage esperada: 5-30% de 570k. Si train_engine < 30k, avisa antes de entrenar.


In [ ]:
# ENGINE PILOTO (3 epocas). Mira loss_mse: objetivo < 0.5 (vs 0.78 con game-result).
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Engine-value pilot',
              overrides={'epochs': 3, 'batch_size': 128, 'lr': 1e-3,
                         'warmup_ratio': 0.05, 'mask_illegal': True,
                         'run_id': 'engine-v1', 'seed': 42,
                         'resume': True, 'use_amp': True,
                         'value_weight': 1.0,
                         'init_ckpt': 'checkpoints/masked-v1/best_model.pt',
                         'data_path': 'data/processed/train_engine.jsonl',
                         'val_path': 'data/processed/val_engine.jsonl'},
              auto=True)
print('mode:', r['mode'])
print(r['result'])


In [ ]:
# ENGINE FULL (10 epocas, resume desde el piloto). Re-ejecutable si muere.
from src.graph_agents.llm_agents import run_agent
r = run_agent('trainer', 'Engine-value full',
              overrides={'epochs': 10, 'batch_size': 128, 'lr': 1e-3,
                         'warmup_ratio': 0.05, 'mask_illegal': True,
                         'run_id': 'engine-v1', 'seed': 42,
                         'resume': True, 'use_amp': True,
                         'value_weight': 1.0,
                         'init_ckpt': 'checkpoints/masked-v1/best_model.pt',
                         'data_path': 'data/processed/train_engine.jsonl',
                         'val_path': 'data/processed/val_engine.jsonl'},
              auto=True)
print(r['result'])


In [ ]:
# EVAL con targets engine (incluye pearson_r, la metrica RQ2)
from src.graph_agents.llm_agents import run_agent
print(run_agent('evaluator', 'Evaluate engine-value vs Stockfish targets',
                overrides={'ckpt_path': 'checkpoints/engine-v1/best_model.pt',
                           'val_path': 'data/processed/val_engine.jsonl',
                           'out_path': 'experiments/evaluation_engine-v1.json'},
                auto=True)['result'])


In [ ]:
# DESCARGA FINAL
!tar -czf /content/engine_artifacts.tar.gz -C /content/chess-transformer \
    checkpoints/engine-v1 experiments/training_log_engine-v1.jsonl \
    experiments/training_results_engine-v1.json \
    experiments/evaluation_engine-v1.json data/processed/engine_targets_stats.json 2>/dev/null; true
!ls -lh /content/engine_artifacts.tar.gz
from google.colab import files
files.download('/content/engine_artifacts.tar.gz')
# Ademas descarga manual: checkpoints/engine-v1/best_model.pt
